# Nivel 2 — Solución Programática: LangChain + Gemini + Telegram
Este notebook contiene la migración integral de la arquitectura visual desarrollada originalmente en N8n hacia un entorno puramente programático en Python. El objetivo principal es construir un agente inteligente de orquestación logística capaz de procesar lenguaje natural, gestionar el inventario de productos de forma automatizada y emitir notificaciones transaccionales en tiempo real a través de un bot de Telegram.

### Componentes Clave de la Arquitectura
- **Orquestación y Procesamiento Cognitivo:** Uso de LangChain junto con el modelo `gemini-2.5-flash`.
- **Capa de Persistencia (Base de Datos):** Simulación relacional mediante hojas estructuradas (`Stock`, `Pedidos`, `Facturas`) dentro de un archivo maestro de Excel.
- **Interfaz de Usuario:** Canal de mensajería asíncrono implementado mediante la API de Telegram.

## 1. Configuración del Entorno y Autenticación
En esta celda se realiza la instalación de las dependencias fundamentales del ecosistema de IA generativa (`langchain-google-genai`), el framework de mensajería (`python-telegram-bot`) y las herramientas de análisis de datos. Adicionalmente, se configuran de forma segura las variables de entorno para las credenciales (`GOOGLE_API_KEY` y `TELEGRAM_BOT_TOKEN`) empleando el sistema de almacenamiento seguro (Secrets) del entorno de ejecución.

In [ ]:
!pip install -q -U langchain langchain-core langchain-google-genai python-telegram-bot pandas==2.2.2 openpyxl pydantic==2.12.3
print("✅ Dependencias instaladas")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 550.1/550.1 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.8/68.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 745.4/745.4 kB 27.9 MB/s eta 0:00:00
✅ Dependencias instaladas


In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    TELEGRAM_BOT_TOKEN = userdata.get("TELEGRAM_BOT_TOKEN")
    print("✅ GOOGLE_API_KEY cargada desde Colab Secrets")
    print("✅ TELEGRAM_BOT_TOKEN cargado desde Colab Secrets")
except Exception:
    TELEGRAM_BOT_TOKEN = os.getenv("TELEGRAM_BOT_TOKEN", "")
    if os.getenv("GOOGLE_API_KEY"):
        print("✅ GOOGLE_API_KEY cargada desde variables de entorno")
    else:
        print("⚠️ Falta GOOGLE_API_KEY")
    if TELEGRAM_BOT_TOKEN:
        print("✅ TELEGRAM_BOT_TOKEN cargado desde variables de entorno")
    else:
        print("⚠️ Falta TELEGRAM_BOT_TOKEN")

✅ GOOGLE_API_KEY cargada desde Colab Secrets
✅ TELEGRAM_BOT_TOKEN cargado desde Colab Secrets


## 2. Carga y Estructuración de Datos (Pandas DataFrames)
El siguiente bloque inicializa la persistencia de datos leyendo el archivo maestro de transacciones `Registros.xlsx`. Utilizando la librería `Pandas`, la información es segmentada en tres estructuras independientes orientadas a subprocesos logísticos específicos: control de existencias actuales (`Stock`), trazabilidad de órdenes de clientes (`Pedidos`) e historial contable (`Facturas`). Al final, se imprime una vista previa para confirmar la integridad de los datos cargados.

In [ ]:
from pathlib import Path
import pandas as pd
from datetime import datetime
from uuid import uuid4

EXCEL_PATH = Path("Registros.xlsx")
TZ_LABEL = "America/Bogota"

assert EXCEL_PATH.exists(), f"No se encontró {EXCEL_PATH.resolve()}"
print(f"✅ Archivo encontrado: {EXCEL_PATH.resolve()}")

✅ Archivo encontrado: /content/Registros.xlsx


In [ ]:
stock_df = pd.read_excel(EXCEL_PATH, sheet_name="Stock")
pedidos_df = pd.read_excel(EXCEL_PATH, sheet_name="Pedidos")
facturas_df = pd.read_excel(EXCEL_PATH, sheet_name="Facturas")

print("Stock:", stock_df.shape)
print("Pedidos:", pedidos_df.shape)
print("Facturas:", facturas_df.shape)

stock_df.head()

Stock: (30, 4)
Pedidos: (0, 11)
Facturas: (0, 6)


,producto_id,descripcion_producto,stock,precio_unitario
0,PROD-001,Caja de guantes industriales talla M,120,18000
1,PROD-002,Cinta de embalaje 48 mm x 100 m,85,9500
2,PROD-003,Rollo de etiqueta térmica 100x100,40,32000
3,PROD-004,Lector de código de barras inalámbrico,12,145000
4,PROD-005,Impresora térmica de etiquetas,6,420000


## 3. Funciones auxiliares para Excel

Estas funciones simulan las tools del agente sobre el archivo `Registros.xlsx`.

In [ ]:
EXPECTED_PEDIDOS = [
    "order_id", "cliente", "chat_id", "producto_id", "descripcion_producto",
    "cantidad", "estado", "stock", "fecha_pedido", "fecha_despacho", "total"
]
EXPECTED_STOCK = ["producto_id", "descripcion_producto", "stock", "precio_unitario"]
EXPECTED_FACTURAS = ["factura_id", "order_id", "cliente", "monto_total", "fecha_factura", "estado_factura"]


def now_str():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


def load_sheets():
    stock = pd.read_excel(EXCEL_PATH, sheet_name="Stock")
    pedidos = pd.read_excel(EXCEL_PATH, sheet_name="Pedidos")
    facturas = pd.read_excel(EXCEL_PATH, sheet_name="Facturas")

    for col in EXPECTED_PEDIDOS:
        if col not in pedidos.columns:
            pedidos[col] = ""
    for col in EXPECTED_STOCK:
        if col not in stock.columns:
            stock[col] = ""
    for col in EXPECTED_FACTURAS:
        if col not in facturas.columns:
            facturas[col] = ""

    return stock[EXPECTED_STOCK].copy(), pedidos[EXPECTED_PEDIDOS].copy(), facturas[EXPECTED_FACTURAS].copy()


def save_sheets(stock, pedidos, facturas):
    with pd.ExcelWriter(EXCEL_PATH, engine="openpyxl", mode="w") as writer:
        pedidos.to_excel(writer, sheet_name="Pedidos", index=False)
        stock.to_excel(writer, sheet_name="Stock", index=False)
        facturas.to_excel(writer, sheet_name="Facturas", index=False)

## 4. Definición de Herramientas del Sistema (Tools)
Para que el agente inteligente interactúe de forma efectiva con el inventario, encapsulamos las reglas lógicas de negocio en funciones deterministas de Python. Aquí se implementan los mecanismos de lectura y escritura segura en el archivo Excel, y se definen las tres herramientas operativas básicas que invocará la IA de forma autónoma:
1. `get_stock`: Verificación en tiempo real de unidades disponibles de un producto.
2. `create_order`: Deducción de inventario, asentamiento del pedido en estado despachado y generación automática de la factura.
3. `get_order_status`: Consulta histórica de estados filtrada por el identificador único de chat del cliente.

In [ ]:
from typing import Optional, Literal
from pydantic import BaseModel, Field

class IntentOutput(BaseModel):
    intencion: Literal["crear_pedido", "consultar_estado", "saludo", "otro"]
    order_id: str = ""
    producto_id: str = ""
    cantidad: int = 0


def get_stock(producto_id: str) -> dict:
    stock, _, _ = load_sheets()
    row = stock[stock["producto_id"].astype(str).str.upper() == producto_id.upper()]
    if row.empty:
        return {"found": False, "producto_id": producto_id, "mensaje": "Producto no encontrado"}
    r = row.iloc[0]
    return {
        "found": True,
        "producto_id": str(r["producto_id"]),
        "descripcion_producto": str(r["descripcion_producto"]),
        "stock": int(r["stock"]),
        "precio_unitario": float(r["precio_unitario"]),
    }


def create_order(cliente: str, chat_id: str, producto_id: str, cantidad: int) -> dict:
    stock, pedidos, facturas = load_sheets()
    info = get_stock(producto_id)
    order_id = f"{cliente}{str(uuid4())[:8]}"
    fecha_pedido = now_str()

    if not info["found"]:
        new_row = {
            "order_id": order_id,
            "cliente": cliente,
            "chat_id": str(chat_id),
            "producto_id": producto_id,
            "descripcion_producto": "",
            "cantidad": int(cantidad),
            "estado": "PRODUCTO_NO_ENCONTRADO",
            "stock": 0,
            "fecha_pedido": fecha_pedido,
            "fecha_despacho": "",
            "total": 0,
        }
        pedidos = pd.concat([pedidos, pd.DataFrame([new_row])], ignore_index=True)
        save_sheets(stock, pedidos, facturas)
        return {"ok": False, "order_id": order_id, "estado": "PRODUCTO_NO_ENCONTRADO", "mensaje": "Producto no encontrado"}

    stock_actual = int(info["stock"])
    precio_unitario = float(info["precio_unitario"])
    total = int(cantidad * precio_unitario)

    if stock_actual >= cantidad:
        stock.loc[stock["producto_id"].astype(str).str.upper() == producto_id.upper(), "stock"] = stock_actual - cantidad
        fecha_despacho = now_str()
        new_row = {
            "order_id": order_id,
            "cliente": cliente,
            "chat_id": str(chat_id),
            "producto_id": producto_id,
            "descripcion_producto": info["descripcion_producto"],
            "cantidad": int(cantidad),
            "estado": "DESPACHADO",
            "stock": stock_actual,
            "fecha_pedido": fecha_pedido,
            "fecha_despacho": fecha_despacho,
            "total": total,
        }
        pedidos = pd.concat([pedidos, pd.DataFrame([new_row])], ignore_index=True)
        factura = {
            "factura_id": f"FAC-{order_id}",
            "order_id": order_id,
            "cliente": cliente,
            "monto_total": total,
            "fecha_factura": fecha_despacho,
            "estado_factura": "GENERADA",
        }
        facturas = pd.concat([facturas, pd.DataFrame([factura])], ignore_index=True)
        save_sheets(stock, pedidos, facturas)
        return {
            "ok": True,
            "order_id": order_id,
            "estado": "DESPACHADO",
            "producto_id": producto_id,
            "descripcion_producto": info["descripcion_producto"],
            "cantidad": int(cantidad),
            "stock_disponible": stock_actual,
            "precio_unitario": precio_unitario,
            "total": total,
            "fecha_pedido": fecha_pedido,
            "fecha_despacho": fecha_despacho,
        }

    new_row = {
        "order_id": order_id,
        "cliente": cliente,
        "chat_id": str(chat_id),
        "producto_id": producto_id,
        "descripcion_producto": info["descripcion_producto"],
        "cantidad": int(cantidad),
        "estado": "SIN_STOCK",
        "stock": stock_actual,
        "fecha_pedido": fecha_pedido,
        "fecha_despacho": now_str(),
        "total": 0,
    }
    pedidos = pd.concat([pedidos, pd.DataFrame([new_row])], ignore_index=True)
    save_sheets(stock, pedidos, facturas)
    return {
        "ok": False,
        "order_id": order_id,
        "estado": "SIN_STOCK",
        "producto_id": producto_id,
        "descripcion_producto": info["descripcion_producto"],
        "cantidad": int(cantidad),
        "stock_disponible": stock_actual,
        "total": 0,
    }


def get_order_status(chat_id: str, order_id: Optional[str] = None) -> dict:
    _, pedidos, _ = load_sheets()
    pedidos["chat_id"] = pedidos["chat_id"].astype(str)
    rows = pedidos[pedidos["chat_id"] == str(chat_id)].copy()
    if order_id:
        rows = rows[rows["order_id"].astype(str) == str(order_id)]
    rows = rows[rows["estado"].astype(str) != "SIN_STOCK"]
    if rows.empty:
        return {"ok": False, "mensaje": "No encontré pedidos válidos para consultar el estado."}
    last = rows.iloc[-1].to_dict()
    return {"ok": True, **last}

## 5. Integración del LLM y Extracción Sintáctica Estructurada
Aquí configuramos la capa de razonamiento del asistente utilizando el modelo `gemini-2.5-flash`. Mediante el uso de esquemas de validación estrictos y la función `with_structured_output` de LangChain, obligamos al modelo lingüístico a transformar los mensajes libres en lenguaje natural de los usuarios a un formato JSON predecible. Esto permite extraer con absoluta precisión variables cruciales como la intención del usuario, el código del producto y la cantidad solicitada, eliminando la ambigüedad semántica.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
)

structured_llm = llm.with_structured_output(IntentOutput)

prompt = ChatPromptTemplate.from_messages([
    ("system", """
Eres un asistente de logística que recibe mensajes por Telegram.
Debes identificar la intención del usuario y devolver únicamente un objeto estructurado.

Reglas:
- intencion solo puede ser: crear_pedido, consultar_estado, saludo, otro.
- Si el usuario quiere comprar o pedir un producto, usa crear_pedido.
- Si el usuario pregunta por el estado de un pedido, usa consultar_estado.
- Si solo saluda, usa saludo.
- Si no aplica ninguna, usa otro.
- producto_id debe verse como PROD-XXX si aparece.
- cantidad debe ser numérica.
- Si falta información, deja el campo vacío o en 0.
"""),
    ("human", "Mensaje del usuario: {mensaje}")
])

classifier_chain = prompt | structured_llm

## 6. Pipeline de Orquestación y Enrutamiento Lógico
La función principal `process_message` actúa como el despachador central de todo el sistema. Este bloque toma la entrada de texto enviada por el cliente, delega la interpretación al modelo Gemini y, mediante un árbol de decisiones lógico, ejecuta la herramienta programática adecuada según la intención identificada (comprar o consultar). Finalmente, consolida los resultados del backend y construye una respuesta clara, amigable y contextualizada para el usuario.

In [ ]:
def build_response(parsed: IntentOutput, cliente: str, chat_id: str) -> str:
    if parsed.intencion == "saludo":
        return (
            "Hola, soy tu asistente de logística. "
            "Puedo ayudarte a crear pedidos y consultar el estado de tus pedidos."
        )

    if parsed.intencion == "crear_pedido":
        if not parsed.producto_id or int(parsed.cantidad or 0) <= 0:
            return (
                "Para crear el pedido necesito un producto y una cantidad. "
                "Ejemplo: quiero pedir PROD-003 cantidad 2"
            )
        result = create_order(cliente=cliente, chat_id=chat_id, producto_id=parsed.producto_id, cantidad=int(parsed.cantidad))
        if result.get("estado") == "DESPACHADO":
            return (
                f"""Tu pedido {result['order_id']} fue procesado correctamente.
Producto: {result['descripcion_producto']}
Cantidad: {result['cantidad']}
Total: {result['total']}
Estado: DESPACHADO"""
            )
        if result.get("estado") == "SIN_STOCK":
            return (
                f"""Tu pedido {result['order_id']} no pudo procesarse.
Producto: {result['descripcion_producto']}
Cantidad solicitada: {result['cantidad']}
Stock disponible: {result['stock_disponible']}
Estado: SIN_STOCK"""
            )
        return "No se encontró el producto solicitado."

    if parsed.intencion == "consultar_estado":
        result = get_order_status(chat_id=chat_id, order_id=parsed.order_id or None)
        if not result.get("ok"):
            return result["mensaje"]
        return (
            f"""Pedido {result['order_id']}
Producto: {result.get('descripcion_producto') or result.get('producto_id')}
Cantidad: {result['cantidad']}
Estado: {result['estado']}
Total: {result.get('total', 0)}
Fecha pedido: {result.get('fecha_pedido', 'No registrada')}
Fecha despacho: {result.get('fecha_despacho', 'Pendiente')}"""
        )

    return (
        """No entendí tu solicitud. Puedes escribir algo como:
- quiero pedir PROD-002 cantidad 2
- consultar estado de mi pedido"""
    )


def process_message(mensaje: str, cliente: str, chat_id: str =):
    parsed = classifier_chain.invoke({"mensaje": mensaje})
    answer = build_response(parsed, cliente=cliente, chat_id=str(chat_id))
    return parsed, answer


## 7. Entorno de Pruebas Unitarias Locales
Antes de exponer la aplicación al entorno de producción o de red externa, esta celda simula la interacción directa de un usuario enviando dos casos de uso típicos en modo local: una orden de compra directa y una solicitud subsiguiente de estado de cuenta. Esto permite verificar la traza interna del agente, validar la correcta ejecución de las consultas sobre el archivo Excel y pulir las respuestas en consola.

In [ ]:
parsed, answer = process_message("quiero pedir PROD-003 cantidad 2", cliente="test_cliente", chat_id="test_chat_id")
print(parsed)
print("---")
print(answer)

intencion='crear_pedido' order_id='' producto_id='PROD-003' cantidad=2
---
Tu pedido Danieldcc5d004 fue procesado correctamente.
Producto: Rollo de etiqueta térmica 100x100
Cantidad: 2
Total: 64000
Estado: DESPACHADO


In [ ]:
parsed, answer = process_message("consultar estado de mi pedido", cliente="test_cliente", chat_id="test_chat_id")
print(parsed)
print("---")
print(answer)

intencion='consultar_estado' order_id='' producto_id='' cantidad=0
---
Pedido Danieldcc5d004
Producto: Rollo de etiqueta térmica 100x100
Cantidad: 2
Estado: DESPACHADO
Total: 64000
Fecha pedido: 2026-06-08 19:18:26
Fecha despacho: 2026-06-08 19:18:26


## 8. Integración de la Interfaz y Despliegue del Bot
Esta sección configura el backend asíncrono que vincula nuestro orquestador programático con los servidores de Telegram. Se definen los manejadores de eventos (`CommandHandler` y `MessageHandler`) para procesar las interacciones entrantes de manera concurrente, extraer los metadatos dinámicos del usuario (nombre y `chat_id`) e iniciar el bucle de escucha activa mediante `start_polling`. Al ejecutar este bloque, el asistente automatizado queda en línea y respondiendo en tiempo real.

In [ ]:
from telegram import Update
from telegram.ext import ApplicationBuilder, CommandHandler, ContextTypes, MessageHandler, filters

async def start_command(update: Update, context: ContextTypes.DEFAULT_TYPE):
    await update.message.reply_text(
        "Hola, soy tu asistente de logística. Puedes escribir: quiero pedir PROD-003 cantidad 2"
    )

async def handle_message(update: Update, context: ContextTypes.DEFAULT_TYPE):
    if not update.message or not update.message.text:
        return

    mensaje = update.message.text
    cliente = update.effective_user.first_name or "Cliente"
    chat_id = str(update.effective_chat.id)

    try:
        parsed, answer = process_message(mensaje=mensaje, cliente=cliente, chat_id=chat_id)
        print("\n===== TRAZA DEL AGENTE =====")
        print("Mensaje:", mensaje)
        print("Estructura detectada:", parsed.model_dump())
        print("Respuesta final:", answer)
        await update.message.reply_text(answer)
    except Exception as e:
        await update.message.reply_text(f"Ocurrió un error procesando tu solicitud: {e}")


def build_telegram_app():
    app = ApplicationBuilder().token(TELEGRAM_BOT_TOKEN).build()
    app.add_handler(CommandHandler("start", start_command))
    app.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, handle_message))
    return app

In [ ]:
app = build_telegram_app()
await app.initialize()
await app.start()
await app.updater.start_polling()

print("✅ Bot corriendo. Ahora envíale un mensaje desde Telegram.")

✅ Bot corriendo. Ahora envíale un mensaje desde Telegram.


In [ ]:
await app.updater.stop()
await app.stop()
await app.shutdown()

print("🛑 Bot detenido.")

🛑 Bot detenido.
